# RIN — эксперимент 5: write-path (event_log -> гейт -> коммит -> forensic)

Механика №2, F_stage1.tex pipeline v0. Кандидаты заданы вручную (не через NLP-экстрактор — см. `experiments/experiment_5.md` про то, почему это осознанный выбор для первого среза), чтобы проверить именно гейт/коммит/forensic-поиск, а не извлечение.

**Сценарий «Азоры»:** Ярослав купил плавки и зонтик для поездки на Азоры 9 сентября. Проверяем: (1) события корректно коммитятся как узлы с правильными полями, (2) дубликат события — пропускается (`epistemic`-гейт), не плодит копию, (3) forensic-поиск по времени находит всё, что случилось «9-го», точным метаданным-совпадением, не диффузией.

In [1]:
import sys
from pathlib import Path

def find_repo_root(marker="requirements.txt"):
    path = Path.cwd()
    for candidate in [path, *path.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"could not find repo root (looked for {marker})")

REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [2]:
%load_ext autoreload
%autoreload 2

## Событие 1: купил плавки

In [3]:
from memory.graph import MemoryGraph
from memory.write_path import Event, Candidate, process_event, find_by_time, find_by_event

G = MemoryGraph()

event_1 = Event(id=0, timestamp=9, source="user:yaroslav",
                 text="Ярослав купил плавки для поездки на Азоры, вылет 9 сентября")
candidates_1 = [
    Candidate(content="yaroslav owns swim trunks", node_kind="relationship_state",
              edges=[("RelatedTo", "azores trip")]),
    Candidate(content="azores trip", node_kind="episode", edges=[]),
]
decisions_1 = process_event(G, event_1, candidates_1)
for d in decisions_1:
    print(f"{'COMMIT' if d.committed else 'SKIP  '} node={d.node_id} :: {d.candidate.content!r} :: {d.reason}")

COMMIT node=0 :: 'yaroslav owns swim trunks' :: passed all gates
COMMIT node=1 :: 'azores trip' :: passed all gates


## Событие 2: то же самое ещё раз (регресс на дубликат — должно пропуститься)

In [4]:
event_2 = Event(id=1, timestamp=9, source="user:yaroslav",
                 text="Ярослав купил плавки для поездки на Азоры, вылет 9 сентября")
candidates_2 = [
    Candidate(content="yaroslav owns swim trunks", node_kind="relationship_state",
              edges=[("RelatedTo", "azores trip")]),
]
decisions_2 = process_event(G, event_2, candidates_2)
for d in decisions_2:
    print(f"{'COMMIT' if d.committed else 'SKIP  '} node={d.node_id} :: {d.candidate.content!r} :: {d.reason}")
print()
print("ОЖИДАЕМ: SKIP, epistemic — дубликат, узел не размножился")
print("total nodes:", len(G.nodes))

SKIP   node=None :: 'yaroslav owns swim trunks' :: epistemic: duplicate of node 0

ОЖИДАЕМ: SKIP, epistemic — дубликат, узел не размножился
total nodes: 2


## Событие 3: купил зонтик (новый факт, та же поездка)

In [5]:
event_3 = Event(id=2, timestamp=9, source="user:yaroslav", text="и ещё зонтик прихватил")
candidates_3 = [
    Candidate(content="yaroslav owns umbrella", node_kind="relationship_state",
              edges=[("RelatedTo", "azores trip")]),
]
decisions_3 = process_event(G, event_3, candidates_3)
for d in decisions_3:
    print(f"{'COMMIT' if d.committed else 'SKIP  '} node={d.node_id} :: {d.candidate.content!r} :: {d.reason}")

COMMIT node=2 :: 'yaroslav owns umbrella' :: passed all gates


## Событие 4: другой день — не должно попасть в forensic-поиск по «9-му»

In [6]:
event_4 = Event(id=3, timestamp=15, source="user:yaroslav", text="купил новый ноутбук")
candidates_4 = [Candidate(content="yaroslav owns laptop", node_kind="relationship_state")]
process_event(G, event_4, candidates_4)
print("готово")

готово


## Forensic-поиск: «что случилось 9-го» — точным совпадением по времени, не диффузией

In [7]:
hits = find_by_time(G, 9)
print(f"узлов с timestamp=9: {len(hits)}")
for n in hits:
    print(f"  [{n.id}] {n.content!r} kind={n.node_kind} conf={n.confidence} prov={n.provenance}")

print()
print("ОЖИДАЕМ: laptop (timestamp=15) — НЕ в списке")
print("laptop present:", any(n.content == "yaroslav owns laptop" for n in hits))

узлов с timestamp=9: 3
  [0] 'yaroslav owns swim trunks' kind=relationship_state conf=0.9 prov=[0]
  [1] 'azores trip' kind=episode conf=0.9 prov=[0]
  [2] 'yaroslav owns umbrella' kind=relationship_state conf=0.9 prov=[2]

ОЖИДАЕМ: laptop (timestamp=15) — НЕ в списке
laptop present: False


## Инспектор: полный дамп графа (узлы + рёбра + провенанс)

In [8]:
for node_id, node in sorted(G.nodes.items()):
    print(f"[{node_id}] {node.content!r}  kind={node.node_kind}  conf={node.confidence}  "
          f"time={node.time}  provenance={node.provenance}")
    for edge in G.out_edges(node_id):
        print(f"      --{edge.type}--> {G.nodes[edge.dst].content!r}")

[0] 'yaroslav owns swim trunks'  kind=relationship_state  conf=0.9  time=9  provenance=[0]
      --RelatedTo--> 'azores trip'
[1] 'azores trip'  kind=episode  conf=0.9  time=9  provenance=[0]
[2] 'yaroslav owns umbrella'  kind=relationship_state  conf=0.9  time=9  provenance=[2]
      --RelatedTo--> 'azores trip'
[3] 'yaroslav owns laptop'  kind=relationship_state  conf=0.9  time=15  provenance=[3]
